In [ ]:
"""
Wind-forcing preparation for the NPD_eORCA025 wind-perturbation attribution experiments
(ANWSUP, ANWTRN-JRA, ANWTRN-HG3, WNDREP-HG3, WNDREP-HG3c), built from JRA55-do winds
(2013, 2023) and HadGEM3-GC31-HH control-1950 winds (model year 1995).

Processing in this script includes:
-------------------------------------
1. Building climatological baseline averages of 10 m zonal (uas) and meridional (vas) winds
from 3-hourly yearly files: JRA55-do (1958-2023) and BASELINE_HG3 (HadGEM3-GC31-HH control-1950,
1976-2005). Leap days are dropped first.

2. Calculating the climatological 'correction factors' (= BASELINE_HG3 - JRA55-do) for uas
and vas, used to remove the mean state bias between the two wind products and correct the wind 
fields for use in WNDREP-HG3c.

3. Creating the perturbed uas forcing files by isolating anomalous westerlies (zonal wind anomalies
exceeding mean + 1 standard deviation over the Central and Eastern Pacific, 10S-10N,
115E-285E): suppressing them in 2023 (ANWSUP), or transplanting them from 2023 JRA55-do
(ANWTRN-JRA) or HG3-1995 (ANWTRN-HG3) onto 2013 JRA55-do winds.

4. Creating the global wind-replacement files for 2013: WNDREP-HG3 (JRA55-do uas/vas replaced
by HG3-1995 winds) and WNDREP-HG3c (same, but with the correction factors removed).

-------------------------------------------------------------------------------------------
Author: Sreevathsa G. (sg13n23@soton.ac.uk; ORCID ID: 0000-0003-4084-9677)
Last updated: 21 September 2026
"""

In [ ]:
# Import statements
import os
import glob
import shutil
import xarray as xr
import numpy as np
from tqdm.notebook import tqdm

In [ ]:
# Setup paths
PARENT_PATH = '/dssgfs01/scratch/sg13n23/ATTRIBUTION_EXPS/NOC_Near_Present_Day/nemo/data/INPUT_eORCA025/JRA55/'
BASE_DIR   = '/dssgfs01/scratch/sg13n23/ATTRIBUTION_EXPS/NPD_Wind_Exp_Diagnostics/'

# PARENT_PATH is where JRA55-do forcing files are stored as {var}_y{year}.nc, e.g., uas_y2013.nc, vas_y2023.nc, etc.

# Making a copy of the control run zonal wind files for year 2013 and 2023 i.e., unperturbed JRA55 winds
shutil.copy(src = PARENT_PATH+'uas_y2013.nc', dst = './data/WND_FLDS/uas_CTRL_2013_y2013.nc')
shutil.copy(src = PARENT_PATH+'uas_y2023.nc', dst = './data/WND_FLDS/uas_CTRL_2023_y2023.nc')

In [ ]:
# =============================================================================
# CALCULATING CLIMATOLOGICAL UAS/VAS BASELINE AVERAGES FOR JRA55-do (1958-2023)
# AND HADGEM3-GC31-HH CONTROL-1950 SIMULATION (BASELINE_HG3: 1976-2005)
# =============================================================================

# JRA55-do 3-hourly data in yearly files: average 1958-2023, dropping leap day first
# ----------------------------------------------------------------------------------
# Create directory for storing climatolgoical baseline files if it doesn't already exist.
os.makedirs('./data/WND_FLDS/BASELINES/', exist_ok=True)
# List of uas/vas files for years 1958-2023 (inclusive), for climatological baseline calculations.
fnames_jra55 = {'uas': [PARENT_PATH + f'uas_y{year}.nc' for year in range(1958, 2023+1)],
                'vas': [PARENT_PATH + f'vas_y{year}.nc' for year in range(1958, 2023+1)]}

# Loop over the two variables (uas and vas) to calculate their respective clim. means.
for var, files, in fnames_jra55.items(): 
    for f in tqdm(files): # Loop over the files
        ds = xr.open_dataset(f)[var]
        # Removing the leap day for time step consistency across all years before averaging.
        ds = ds.where(~((ds.time.dt.month == 2) & (ds.time.dt.day == 29)), drop = True)
        if files.index(f) == 0: # If it's the first file, initialize the mean dataset with this dataset
            mean = ds
        else: 
            # For subsequent files, add the dataset to the mean after making the time coordinate 
            # consistent across teh datasets being added, ensuring time axis alignment.
            ds['time'] = mean['time']
            mean = mean + ds
    mean = mean / len(fnames_jra55[var]) # Mean over the number of files
    # Save the climatological mean to a NetCDF file after converting it to a xr.Dataset.
    mean.to_dataset().to_netcdf(f'./data/WND_FLDS/BASELINES/{var}_JRA55_y1958-2023.nc')


# BASELINE_HG3 3-hourly data in yearly files: average 1976-2005, dropping leap day first
# ---------------------------------------------------------------------------------------
# List of uas/vas files of HadGEM3-GC31-HH (control-1950) of CMIP6 HighResMIP, for years 1976-2005 (inclusive), for climatological baseline calculations.
fnames_hg3 = {'uas': [BASE_DIR + f'HadGEM3-GC31-HH/{year}/HadGEM3-GC31-HH_uas_control-1950_3hr_y{year}.nc' for year in range(1976, 2006)],
              'vas': [BASE_DIR + f'HadGEM3-GC31-HH/{year}/HadGEM3-GC31-HH_vas_control-1950_3hr_y{year}.nc' for year in range(1976, 2006)]}
'''
Preprocessing note: The above files were created from the original model output by bilinearly interpolating (using CDO operators) from the
model grid (HadGEM3-GC31-HH control-1950) to the JRA55 grid, so they can be used to force NEMO. We converted the model's 360-day calendar 
to a Gregorian calendar by removing 30 February and interpolating onto a regular 3-hourly time axis with nearest-neighbour interpolation, 
which fills the missing 31st day of 31-day months from the nearest available time steps. Leap years therefore retain 29 February, which is 
dropped in the averaging below so that every year has 2920 time steps. Furthermore, we merged files for each year into a single file. These 
steps were performed in a different HPC than where this repository was developed; please reach out to the author if you have any questions 
about these preprocessing steps.
'''

for var, files, in fnames_hg3.items(): # Loop over the two variables (uas and vas) to calculate their climatological means.
    for f in tqdm(files): # Loop over the files
        # Open the dataset and extract the 'uas' or 'vas' variable.
        ds = xr.open_dataset(f)[var]
        # Removing the leap day for time step consistency across all years before averaging.
        ds = ds.where(~((ds.time.dt.month == 2) & (ds.time.dt.day == 29)), drop = True)
        if files.index(f) == 0: # If it's the first file, initialize the mean dataset with this dataset
            mean = ds
        else: 
            # For subsequent files, add the dataset to the mean after making the time coordinate 
            # consistent across teh datasets being added, ensuring time axis alignment.
            ds['time'] = mean['time']
            mean = mean + ds
    mean = mean / len(fnames_hg3[var]) # Mean over the number of files
    # Save the climatological mean to a NetCDF file after converting it to a xr.Dataset.
    mean.to_dataset().to_netcdf(f'./data/WND_FLDS/BASELINES/{var}_BASELINE_HG3_y1976-2005.nc')

In [ ]:
# ======================================================================================================
# CALCULATING CLIMATOLOGICAL 'CORRECTION' FACTORS FOR BOTH UAS AND VAS TO USE FOR THE WNDREP EXPERIMENTS
# ======================================================================================================

# Loading in the JRA55-do and BASELINE_HG3 climatological means for uas and vas, and calculating the correction factors.

WNDS_JRA55 = {'uas': xr.open_dataset('./data/WND_FLDS/BASELINES/uas_JRA55_y1958-2023.nc')['uas'],
              'vas': xr.open_dataset('./data/WND_FLDS/BASELINES/vas_JRA55_y1958-2023.nc')['vas']}
WNDS_BASELINE_HG3 = {'uas': xr.open_dataset('./data/WND_FLDS/BASELINES/uas_BASELINE_HG3_y1976-2005.nc')['uas'],
                     'vas': xr.open_dataset('./data/WND_FLDS/BASELINES/vas_BASELINE_HG3_y1976-2005.nc')['vas']}

# Calculating the correction factors for uas and vas by subtracting the JRA55-do from the HG3 baseline.
CORRECTION_FACTORS = {}
CORRECTION_FACTORS['uas'] = WNDS_BASELINE_HG3['uas'] - WNDS_JRA55['uas']
CORRECTION_FACTORS['vas'] = WNDS_BASELINE_HG3['vas'] - WNDS_JRA55['vas']

# Making a directory to save the correction factors if it doesn't already exist.
os.makedirs('./data/WND_FLDS/CLIM_CORR/', exist_ok=True)
# Saving the correction factors to NetCDF files  after converting it to a xr.Dataset, for later use in preparing
# surface wind forcing files for the WNDREP experiments (next cell).
CORRECTION_FACTORS['uas'].to_dataset().to_netcdf('./data/WND_FLDS/CLIM_CORR/uas_clim_corr_HG3-JRA.nc')
CORRECTION_FACTORS['vas'].to_dataset().to_netcdf('./data/WND_FLDS/CLIM_CORR/vas_clim_corr_HG3-JRA.nc')

In [ ]:
# ================================================================================================
# WIND PERTURBATIONS TO UAS AND/OR VAS FORCING FILES FOR THE EXPERIMENTS: ANWSUP, ANWTRN, WNDREP
# ================================================================================================

# ------------------------------------------------------------
# ANWSUP - Suppressing anomalous westerlies in an El Nino year
# ------------------------------------------------------------

# Loading in the 2023 JRA55-do uas wind field
ds_uas_y2023 = xr.open_dataset('./data/WND_FLDS/uas_CTRL_2023_y2023.nc')
# Calculating the anomaly of the 2023 uas winds relative to the JRA55-do 1958-2023 mean.
anom_uas_y2023 = (ds_uas_y2023['uas'] - WNDS_JRA55['uas'].values).to_dataset()

# Making a copy of the 2023 JRA55-do uas; modifications ("perturbations") will be made to this copy for the ANWSUP experiment
ds_uas_ANWSUP = ds_uas_y2023['uas'].copy(deep = True)

# Isolating anomalous westerlies in the Central Equatorial Pacific (CEP) region (10S-10N, 115E-285E)
# that exceed the mean + 1 standard deviation of uas anomalies computed over the full CEP region, and setting all other values to zero.
anom_uas_y2023_CEP = anom_uas_y2023.sel(lat = slice(-10,10), lon = slice(115,-75+360))
anom_uas_y2023_CEP['uas'] = anom_uas_y2023_CEP['uas'].where(anom_uas_y2023_CEP['uas']>=(np.nanmean(anom_uas_y2023_CEP['uas']) + np.nanstd(anom_uas_y2023_CEP['uas'])), 0)

# Removing the anomalous westerlies from the 2023 JRA55-do uas wind fields in the CEP region to suppress them.
ds_uas_ANWSUP.loc[dict(lat = slice(-10,10), lon = slice(115,-75+360))] -= anom_uas_y2023_CEP['uas']
# Saving the modified uas wind field to a NetCDF file after converting it to a xr.Dataset, for the ANWSUP experiment.
ds_uas_ANWSUP.to_dataset().to_netcdf('./data/WND_FLDS/uas_ANWSUP_y2023.nc')

# ------------------------------------------------------------------------------------
# ANWTRN - Transplanting anomalous westerlies from an El Niño year into a neutral year
# ------------------------------------------------------------------------------------

# ANWTRN-JRA - Transplanting anomalous westerlies from an El Niño year (2023) into a neutral year (2013) using JRA55-do winds

# Loading in the 2013 JRA55-do uas wind field
ds_uas_y2013 = xr.open_dataset('./data/WND_FLDS/uas_CTRL_2013_y2013.nc')

# Making a copy of the 2023 JRA55-do uas; modifications will be made to this copy for the ANWTRN-JRA experiment
ds_uas_ANWTRN_JRA55 = ds_uas_y2013['uas'].copy(deep = True)

# Adding the anomalous westerlies from the 2023 JRA55-do uas wind fields in the CEP region (isolated above IN ANWSUP part)
# to the 2013 JRA55-do uas wind fields, and therefore transplanting them onto a ENSO-neutral background.
ds_uas_ANWTRN_JRA55.loc[dict(lat = slice(-10,10), lon = slice(115,-75+360))] += anom_uas_y2023_CEP['uas'].values
# Saving the modified uas wind field to a NetCDF file after converting it to a xr.Dataset, for the ANWTRN-JRA experiment.
ds_uas_ANWTRN_JRA55.to_dataset().to_netcdf('./data/WND_FLDS/uas_ANWTRN-JRA_y2013.nc')

# ----
# ANWTRN-HG3 - Transplanting anomalous westerlies from an El Niño year in the HadGEM3-GC31-HH control-1950 coupled model simulation (model year 1995) 
# into a ENSO-neutral year's (2013) JRA55-do uas winds, in a similar way to how the above modifications (for ANWTRN-JRA) were made.

# Making a copy of the HG3-1995 year uas wind files; to extract the anomalous westerlies to transplant onto JRA55 2013 uas winds
shutil.copy(src = BASE_DIR + 'HadGEM3-GC31-HH/1995/HadGEM3-GC31-HH_uas_control-1950_3hr_y1995.nc', dst = './data/WND_FLDS/uas_HG3-1995_y1995.nc')

# Loading in the HG3-1995 year uas wind files and calculating the anomaly relative to the HG3 climatological baseline (BASELINE_HG3).
ds_uas_HG3_1995 = xr.open_dataset('./data/WND_FLDS/uas_HG3_1995_y1995.nc')
anom_uas_HG3_1995 = (ds_uas_HG3_1995['uas'] - WNDS_BASELINE_HG3['uas'].values).to_dataset()

# Making a copy of the 2013 JRA55-do uas; modifications will be made to this copy for the ANWTRN-HG3 experiment
ds_uas_ANWTRN_HG3 = ds_uas_y2013['uas'].copy(deep = True)

# Isolating anomalous westerlies in the Central and Eastern Pacific (CEP) region (10S-10N, 115E-285E) from HG3-1995
# similar to how it was done for the ANWSUP experiment above (i.e., only retaining values that exceed the mean + 1 standard 
# deviation of uas anomalies computed over the CEP region, and setting all other values to zero).
anom_uas_HG3_1995_CEP = anom_uas_HG3_1995.sel(lat = slice(-10,10), lon = slice(115,-75+360))
anom_uas_HG3_1995_CEP['uas'] = anom_uas_HG3_1995_CEP['uas'].where(anom_uas_HG3_1995_CEP['uas']>=(np.nanmean(anom_uas_HG3_1995_CEP['uas']) + np.nanstd(anom_uas_HG3_1995_CEP['uas'])), 0)

# Adding the anomalous westerlies from the HG3-1995 uas wind fields in the CEP region to 
# the 2013 JRA55-do uas wind fields, and therefore transplanting them onto a ENSO-neutral background.
ds_uas_ANWTRN_HG3.loc[dict(lat = slice(-10,10), lon = slice(115,-75+360))] += anom_uas_HG3_1995_CEP['uas'].values
# Saving the modified uas wind fields to a NetCDF file after converting it to a xr.Dataset, for the ANWTRN-HG3 experiment.
ds_uas_ANWTRN_HG3.to_dataset().to_netcdf('./data/WND_FLDS/uas_ANWTRN-HG3_y2013.nc')

# ---------------------------------------------------------------------------------
# WNDREP - Global Replacement of JRA55-do winds with coupled model winds (HG3-1995)
# ---------------------------------------------------------------------------------

# WNDREP-HG3 - Replacing JRA55-do winds with HadGEM3-GC31-HH control-1950 winds (HG3-1995) for the year 2013

ds_uas_HG3_1995 =xr.open_dataset(BASE_DIR + 'HadGEM3-GC31-HH/1995/HadGEM3-GC31-HH_uas_control-1950_3hr_y1995.nc')
# Resetting the time coordinate of the HG3-1995 uas dataset to match that of the 2013 JRA55-do uas dataset
ds_uas_HG3_1995['time'] = xr.open_dataset(PARENT_PATH + f'uas_y2013.nc')['time']
ds_uas_HG3_1995.to_netcdf('./data/WND_FLDS/uas_WNDREP-HG3_y2013.nc') # Saving the file

ds_vas_HG3_1995 =xr.open_dataset(BASE_DIR + 'HadGEM3-GC31-HH/1995/HadGEM3-GC31-HH_vas_control-1950_3hr_y1995.nc')
# Resetting the time coordinate of the HG3-1995 vas dataset to match that of the 2013 JRA55-do vas dataset
ds_vas_HG3_1995['time'] = xr.open_dataset(PARENT_PATH + f'vas_y2013.nc')['time']
ds_vas_HG3_1995.to_netcdf('./data/WND_FLDS/vas_WNDREP-HG3_y2013.nc') # Saving the file

# ----
# WNDREP-HG3c - Same as above but removing the 'correction factor' calculated as BASELINE_HG3 - JRA55-do for uas and vas

# Removing the correction factors from the HG3-1995 uas and vas datasets to create the WNDREP-HG3c datasets.
ds_uas_corr_HG3_1995 = (ds_uas_HG3_1995['uas'] - CORRECTION_FACTORS['uas'].values).to_dataset()
ds_vas_corr_HG3_1995 = (ds_vas_HG3_1995['vas'] - CORRECTION_FACTORS['vas'].values).to_dataset()
# Saving the corrected datasets to NetCDF files after converting them to a xr.Datasets, for the WNDREP-HG3c experiment.
ds_uas_corr_HG3_1995.to_netcdf('./data/WND_FLDS/uas_WNDREP-HG3c_y2013.nc')
ds_vas_corr_HG3_1995.to_netcdf('./data/WND_FLDS/vas_WNDREP-HG3c_y2013.nc')